# Suite EXP — Expressions

`Expressions.OfLiteral`, `OfOperation`, `OfVariable`, `OfLet`, `OfQuantifier` and `OfAny` follow the builder pattern
(`Data`, `Builder`, `Spec`, `resolve`) and have meta-schemas (`Schema`): ordinary value object schemas, registered as
`Expressions.*` in the dialect's `store`. An expression is a value of its kind's: `reader()` gives it, through which it
serializes, validates and compares like any value, and `DIALECT.term(value)` rebuilds an expression from one.
`Writer`s write expressions with methods. `Evaluators` computes them (suite EVL).

In [ ]:
import math

from mbse.Expressions import Evaluators as V, Expressions as E
from mbse.Expressions.Dialects.Python import Text as PText
from mbse.Expressions.Framework import Terms as F
from mbse.Schemas.Framework import Comparison as C, JSON, Plain, Proxies, Schemas as S, Validators, YAML
from mbse.Schemas.Framework.Errors import DecodeError
from support import raises, text
from mbse.Expressions import register

store = register(Proxies.OfStore())  # holds every dialect's meta-schemas
A = F.ARGUMENT  # a value of any type: an expression, tagged with its kind's meta-schema's name

lit = E.OfLiteral.Builder().value(3).create()
op = E.OfOperation.Builder().name("f").arguments(1, lambda a: a.as_literal("x")).arguments(lit).create()

## EXP-01 · Literals hold one native value

In [ ]:
assert lit.value == 3 and lit.validate() == [] and lit.schema_name() == "Expressions.OfLiteral"
assert E.OfLiteral.resolve(3).value == 3 and E.OfLiteral.resolve(lit) is lit
assert E.OfLiteral.resolve(lambda b: b.value("x")).value == "x"
for value in [0, 2**70, -0.0, math.nan, "", b"", False]:  # every native type
    assert E.OfLiteral.resolve(value).validate() == []
assert E.OfLiteral.Data().validate() == ["a literal needs a value"]
assert E.OfLiteral.Data([1]).validate() == ["a literal must hold a native value, got list"]
# A builder keeps a value that is not native under `value`, for validation to report.
assert E.OfLiteral.Builder(E.OfLiteral.Data([1])).clone().value == [1]
assert E.OfLiteral.Builder().set("value", [2]).create().value == [2]

## EXP-02 · Operations have a name and ordered arguments

In [ ]:
assert op.name == "f" and [a.value for a in op.arguments] == [1, "x", 3] and op.arguments[2] is lit
assert op.validate() == [] and op.schema_name() == "Expressions.OfOperation"
nested = E.OfOperation.resolve(lambda o: o.name("not").arguments(lambda a: a.as_operation(op)))
assert nested.arguments == (op,) and E.OfOperation.resolve(op) is op
assert E.OfOperation.Data().validate() == ["an operation needs a name"]
assert E.OfOperation.Data("").validate() == ["an operation needs a name"]
assert E.OfOperation.Data(3).validate() == ["an operation's name must be a str, got int"]
bad = E.OfOperation.Data("f", (E.OfLiteral.Data(), E.OfOperation.Data(None, (E.OfLiteral.Data([]),)), 7))
assert bad.validate() == [
    "argument 0: a literal needs a value",
    "argument 1: an operation needs a name",
    "argument 1: argument 0: a literal must hold a native value, got list",
    "argument 2: not an expression: 7",
]

## EXP-03 · Cycles are reported, not followed; shared arguments are not cycles

In [ ]:
loop = E.OfOperation.Builder().name("f").create()
E.OfOperation.Builder(loop).arguments(loop).update()
assert loop.validate() == ["argument 0: the expression contains a cycle"]
assert E.OfOperation.Builder().name("g").arguments(op, op).create().validate() == []

## EXP-04 · Builders: create, clone and update; `OfAny` selects a kind; Spec errors

In [ ]:
source = E.OfOperation.Builder().name("f").arguments(1).create()
copy = E.OfOperation.Builder(source).arguments(2).clone()
assert [a.value for a in copy.arguments] == [1, 2] and len(source.arguments) == 1
assert E.OfOperation.Builder(source).name("g").update() is source and source.name == "g"
literal = E.OfLiteral.Builder().value(3).create()
assert E.OfLiteral.Builder(literal).value(4).clone().value == 4 and literal.value == 3
assert E.OfLiteral.Builder(literal).value(5).update() is literal and literal.value == 5
with raises(ValueError, match="create() is only valid without a source instance"):
    E.OfLiteral.Builder(literal).create()
with raises(ValueError, match="clone() is only valid with a source instance"):
    E.OfOperation.Builder().clone()
with raises(ValueError, match="update() is only valid with a source instance"):
    E.OfLiteral.Builder().update()
with raises(TypeError):
    E.OfLiteral.Builder().create(1)
with raises(TypeError, match="expected operation data to build from, got int"):
    E.OfOperation.Builder(3)
with raises(TypeError, match="expected literal data to build from, got str"):
    E.OfLiteral.Builder("x")

assert E.OfAny.Builder().as_literal(1).create().value == 1
assert E.OfAny.resolve(lambda a: a.as_operation(lambda o: o.name("f"))).name == "f"
assert E.OfAny.resolve(2).value == 2 and E.OfAny.resolve(op) is op
with raises(ValueError, match="create() is only valid without a source instance"):
    E.OfAny.Builder(literal).create()
with raises(ValueError, match="clone() is only valid with a source instance"):
    E.OfAny.Builder().clone()
with raises(ValueError, match="update() is only valid with a source instance"):
    E.OfAny.Builder().update()
with raises(ValueError, match="no kind selected"):
    E.OfAny.Builder().create()
cloned = E.OfAny.Builder(literal).clone()
assert cloned is not literal and cloned.value == literal.value
shallow = E.OfAny.Builder(op).clone()
assert shallow is not op and shallow.name == op.name and shallow.arguments == op.arguments
assert E.OfAny.Builder(literal).as_literal(1).clone().value == 1 and literal.value == 5
assert E.OfAny.Builder(literal).as_literal(9).update() is literal and literal.value == 9
target = E.OfOperation.Builder().name("f").create()
assert E.OfAny.Builder(target).as_operation(lambda o: o.name("h").arguments(1)).update() is target
assert target.name == "h" and [a.value for a in target.arguments] == [1]
with raises(TypeError, match="update() cannot change the kind of the source expression"):
    E.OfAny.Builder(literal).as_operation(op).update()

with raises(TypeError, match="a class is not a Spec here, got int"):
    E.OfAny.resolve(int)
with raises(TypeError, match="a Spec callable must return its builder, got None"):
    E.OfAny.resolve(lambda a: None)
with raises(TypeError, match="expected an operation or a callable taking its builder, got 3"):
    E.OfOperation.resolve(3)
with raises(TypeError, match="expected an expression, a native value or a callable taking its builder, got [1]"):
    E.OfAny.resolve([1])
with raises(TypeError, match="expected a native value, a literal or a callable taking its builder, got None"):
    E.OfLiteral.resolve(None)

## EXP-05 · The meta-schemas are ordinary value object schemas, registered in the dialect's store

In [ ]:
kinds = [E.OfLiteral, E.OfOperation, E.OfVariable, E.OfLet, E.OfQuantifier]
for schema in [*(k.Schema for k in kinds), E.OfAny.Schema]:
    assert schema.validate() == []
assert not any(k.Schema.ref for k in kinds) and all(k.Schema.adjacencies == {} for k in kinds)  # values, never linked
assert list(E.OfLiteral.Schema.properties) == ["int", "float", "str", "bool", "bytes", "domain"]
assert list(E.OfOperation.Schema.properties) == ["name", "domain", "arguments"]  # its result's domain, for conversions
assert list(E.OfVariable.Schema.properties) == ["name"] and list(E.OfLet.Schema.properties) == ["name", "arguments"]
assert list(E.OfQuantifier.Schema.properties) == ["name", "quantifier", "arguments"]
arguments = E.OfOperation.Schema.properties["arguments"].type  # a dense container of any dialect's expressions
assert isinstance(arguments, S.OfContainer.Data) and arguments.arrangement == "dense" and arguments.element is A
assert isinstance(A, S.OfAny.Value) and not A.types
assert [b.type for b in E.OfAny.Schema.branches] == [k.Schema for k in kinds]
assert [k.Data.KIND for k in kinds] == ["literal", "operation", "variable", "let", "quantifier"]
names = [E.LITERAL, E.OPERATION, E.VARIABLE, E.LET, E.QUANTIFIER]
assert names == ["Expressions.OfLiteral", "Expressions.OfOperation", "Expressions.OfVariable", "Expressions.OfLet",
                 "Expressions.OfQuantifier"]
assert [store.registered(name) for name in names] == [k.Schema for k in kinds]  # registered in any store, too
assert [E.store.registered(name) for name in names] == [k.Schema for k in kinds]
assert [k.Data().schema_name() for k in kinds] == names and E.store is E.DIALECT.store

## EXP-06 · Round trips through Plain, JSON and YAML rebuild `Data`; an expression is written nested

In [ ]:
expr = E.OfOperation.Builder().name("f").arguments(
    lambda a: a.as_operation(lambda o: o.name("eq").arguments(1, 1.5)), "s", b"\x01", True).create()
expected = JSON.ToJSON(E.store)(A, expr.reader())
for encode, decode in [(Plain.ToPlain(E.store), Plain.FromPlain), (JSON.ToJSON(E.store), JSON.FromJSON), (YAML.ToYAML(E.store), YAML.FromYAML)]:
    back = E.DIALECT.term(decode(E.store)(A, encode(A, expr.reader())))
    assert isinstance(back, E.OfOperation.Data) and back is not expr and F.same(back, expr)
    assert back.validate() == [] and JSON.ToJSON(E.store)(A, back.reader()) == expected
not_ = E.OfOperation.Builder().name("not").arguments(True).create()
assert JSON.ToJSON(E.store)(A, not_.reader()) == (
    '{"Expressions.OfOperation": {"name": "not", "arguments": [{"Expressions.OfLiteral": {"bool": true}}]}}')
assert JSON.ToJSON(E.store)(E.OfOperation.Schema, not_.reader()) == (  # untagged, where its schema is given
    '{"name": "not", "arguments": [{"Expressions.OfLiteral": {"bool": true}}]}')
alone = JSON.ToJSON(E.store)(E.OfLiteral.Schema, E.OfLiteral.resolve(-0.0).reader())
assert alone == '{"float": -0.0}'
assert math.copysign(1.0, E.DIALECT.term(JSON.FromJSON(E.store)(E.OfLiteral.Schema, alone)).value) == -1.0
for kind in kinds:  # nothing set: nothing written
    assert JSON.ToJSON(E.store)(kind.Schema, kind.Data().reader()) == "{}"
    assert F.same(E.DIALECT.term(JSON.FromJSON(E.store)(kind.Schema, "{}")), kind.Data())
gap = E.OfLet.Data("x", None, E.OfLiteral.resolve(1))  # an empty slot is null
assert JSON.ToJSON(E.store)(E.OfLet.Schema, gap.reader()) == (
    '{"name": "x", "arguments": [null, {"Expressions.OfLiteral": {"int": 1}}]}')
assert F.same(E.DIALECT.term(gap.reader()), gap) and E.DIALECT.term(gap.reader()).value is None
with raises(DecodeError, match="$.kind: the value object has no property 'kind'"):
    JSON.FromJSON(E.store)(E.OfLiteral.Schema, '{"kind": "operation"}')
with raises(DecodeError, match="no schema registered as 'Contact'"):
    JSON.FromJSON(E.store)(A, '{"Contact": {}}')

## EXP-07 · An expression's value is built through the dialect's store, as any value is, and rebuilt as an expression

In [ ]:
one = E.store.builder(E.LITERAL).int(1).create()
built = E.store.builder(E.OPERATION).name("f").arguments(
    lambda p: p.as_value(lambda a: a.as_container(lambda c: c.at(0).as_object(one) and c) or a)).create()
assert F.same(E.DIALECT.term(built), E.operation("f", 1).data) and E.DIALECT.term(one).value == 1
assert Plain.ToPlain(E.store)(A, built) == {"Expressions.OfOperation": {"name": "f", "arguments": [{"Expressions.OfLiteral": {"int": 1}}]}}
Contact = S.OfObject.Builder().name("Contact").properties(text("name")).create()
store.register(Contact)
with raises(TypeError, match="not a value of a Basic kind: 'Contact'"):
    E.DIALECT.term(store.Contact().name("Ada").create())

## EXP-08 · Validation and comparison work through the meta-schemas

In [ ]:
validate = Validators.Validate(E.store)
assert validate(E.OfOperation.Schema, expr.reader()) == [] and validate(E.OfLiteral.Schema, lit.reader()) == []
assert validate(E.OfLiteral.Schema, expr.reader()) == [
    "the value is a 'Expressions.OfOperation', not an instance of the given schema"]


def record(schema, expression):
    return C.OfObject(schema, expression.reader(), value=True)


one, two = E.OfLiteral.resolve(1), E.OfLiteral.resolve(2)
assert record(E.OfLiteral.Schema, one).compare(record(E.OfLiteral.Schema, E.OfLiteral.resolve(1))) == 0
assert record(E.OfLiteral.Schema, one).compare(record(E.OfLiteral.Schema, two)) is None  # objects are not ordered
assert record(E.OfLiteral.Schema, one).compare(record(E.OfLiteral.Schema, E.OfLiteral.resolve(1.0))) is None
f1 = E.OfOperation.resolve(lambda o: o.name("f").arguments(1))
f2 = E.OfOperation.resolve(lambda o: o.name("f").arguments(2))
assert record(E.OfOperation.Schema, f1).compare(record(E.OfOperation.Schema, f2)) is None  # its arguments are its value
assert record(E.OfOperation.Schema, f1).compare(record(E.OfOperation.Schema, E.operation("f", 1).data)) == 0

## EXP-09 · An expression its meta-schema does not fit, or one with a cycle, has no value

In [ ]:
with raises(DecodeError, match="$.name: expected str, got int"):
    E.OfOperation.Data(3).reader()
with raises(DecodeError, match="$.name: expected str, got list"):
    E.OfOperation.Data([1]).reader()
with raises(DecodeError, match="$.arguments[0][\"Expressions.OfOperation\"].name: expected str, got int"):
    E.OfOperation.Data("f", (E.OfOperation.Data(3),)).reader()
with raises(TypeError, match="a literal must hold a native value, got list"):
    E.OfLiteral.Data([1]).reader()
with raises(ValueError, match="the expression contains a cycle"):
    loop.reader()
shared = E.literal(1).data
assert JSON.ToJSON(E.store)(E.OfOperation.Schema, E.operation("f", shared, shared).data.reader()) == (
    '{"name": "f", "arguments": [{"Expressions.OfLiteral": {"int": 1}}, {"Expressions.OfLiteral": {"int": 1}}]}')

## EXP-10 · Variables and lets

In [ ]:
v = E.OfVariable.Builder().name("x").create()
assert v.name == "x" and v.schema_name() == "Expressions.OfVariable" and E.OfVariable.resolve("x").name == "x"
assert v.validate() == ["variable 'x' is not bound"] and v.validate(bound={"x"}) == []
assert E.OfVariable.Data().validate() == ["a variable needs a name"]
assert E.OfVariable.Data(1).validate() == ["a variable's name must be a str, got int"]
assert E.OfVariable.resolve(v) is v and E.OfVariable.Builder(v).name("y").clone().name == "y"
let = E.OfLet.Builder().name("x").value(1).body(v).create()
assert let.name == "x" and let.value.value == 1 and let.body is v and let.validate() == []
assert E.OfLet.Builder(let).body(2).clone().body.value == 2 and let.body is v
assert E.OfLet.Builder(let).value(5).update() is let and let.value.value == 5 and let.body is v
assert E.OfLet.resolve(lambda b: b.name("y").value(1).body(2)).name == "y" and E.OfLet.resolve(let) is let
assert E.OfLet.Data("x").validate() == ["a let needs a value", "a let needs a body"]
assert E.OfLet.Data(None, E.OfLiteral.resolve(1), E.OfVariable.resolve("x")).validate() == [
    "a let needs a name", "body: variable 'x' is not bound"]
same_name = E.OfLet.Data("y", E.OfVariable.resolve("y"), E.OfVariable.resolve("y"))
assert same_name.validate() == ["value: variable 'y' is not bound"]  # the name is bound in the body only
assert E.OfAny.resolve(lambda a: a.as_variable("x")).name == "x" and E.OfAny.resolve(lambda a: a.as_let(let)) is let
assert E.OfAny.Builder(v).clone().name == "x" and E.OfAny.Builder(let).clone().body is v
unnamed = [E.OfOperation.Builder().create(), E.OfVariable.Builder().create(), E.OfLet.Builder().create()]
assert [u.name for u in unnamed] == [None, None, None] and unnamed[2].value is None
open_let = E.OfLet.Builder(E.OfLet.Data("x")).clone()
assert open_let.value is None and open_let.body is None
with raises(TypeError, match="expected a let or a callable taking its builder, got 3"):
    E.OfLet.resolve(3)
with raises(TypeError, match="expected a name, a variable or a callable taking its builder, got 3"):
    E.OfVariable.resolve(3)
with raises(TypeError, match="a variable has no variadic arguments"):
    E.OfVariable.Builder().arguments(1)
with raises(KeyError, match="a let has no argument 'other'"):
    E.OfLet.Builder().argument("other", 1)
with raises(KeyError, match="a variable has no attribute 'value'"):
    E.OfVariable.Builder().set("value", 1)

## EXP-11 · Writers build expressions with methods

In [ ]:
this = E.variable("this")
writer = this.age.ge(18)
assert isinstance(writer, E.Writer) and writer.data.name == "ge" and writer.data.arguments[1].value == 18
get = writer.data.arguments[0]
assert get.name == "get" and get.arguments[0] is this.data and get.arguments[1].value == "age"
assert this.get("eq").data.arguments[1].value == "eq"  # a property named like a method
methods = [this.eq(1), this.ne(1), this.lt(1), this.le(1), this.gt(1), this.ge(1), this.and_(True), this.or_(True),
           this.not_(), this.implies(True), this.add(1), this.sub(1), this.mul(1), this.neg(), this.has("x"),
           this.get("x")]
assert [m.data.name for m in methods] == ["eq", "ne", "lt", "le", "gt", "ge", "and", "or", "not", "implies", "add",
                                          "sub", "mul", "neg", "has", "get"]
assert E.literal(3).data.value == 3 and E.operation("in", 1, 2).data.name == "in"
bound = E.let_("a", 1, E.variable("a"))
assert bound.data.name == "a" and E.OfLet.resolve(bound) is bound.data and E.OfOperation.resolve(writer) is writer.data
assert E.OfAny.resolve(writer) is writer.data and E.OfAny.resolve(lambda a: a.as_operation(writer)) is writer.data
with raises(TypeError, match="expected an operation or a callable taking its builder"):
    E.OfOperation.resolve(this)  # a variable, not an operation
with raises(AttributeError):
    this._private  # Python's own probes are not properties

## EXP-12 · `Python.Text.FromFunction` reads an expression from a Python function (Python only)

In [ ]:
import linecache

from mbse.Expressions import Translators as T
from mbse.Expressions.Framework import Terms as F
from mbse.Expressions.Dialects.Python import Expressions as P


def text_of(expression):
    """The JSON of an expression, for comparing two by their content."""
    return JSON.ToJSON(E.store)(A, E.OfAny.resolve(expression).reader())


def same_as(function, writer):
    assert text_of(PText.FromFunction(function)) == text_of(writer), (text_of(PText.FromFunction(function)), text_of(writer))


MIN = 18
this, t, a, one = E.variable("this"), E.variable("t"), E.variable("a"), E.literal(1)
same_as(lambda this: this.age >= MIN and this.email is not None, this.age.ge(18).and_(this.has("email")))
same_as(lambda t: t.a < 1 < t.b, t.a.lt(one).and_(one.lt(t.b)))  # a chain; the middle operand is shared
same_as(lambda t: -t.a * 2 + +t.b - 1 == 0, t.a.neg().mul(2).add(t.b).sub(1).eq(0))
same_as(lambda t: (lambda a: a.b > a.c)(t.d), E.let_("a", t.d, a.b.gt(a.c)))  # a let


def constraint(t):
    """A def works too, if its body is one return."""
    return not hasattr(t, "x") or getattr(t, "y") != 2 or t.z is None


same_as(constraint, t.has("x").not_().or_(t.get("y").ne(2)).or_(t.has("z").not_()))


def plain(t):
    return t.a


same_as(plain, t.a)
first, second = (lambda t: t.a), (lambda t: t.b)  # two on one line
same_as(first, t.a)
same_as(second, t.b)
LIMIT, LIMIT_DATA = E.literal(5), E.OfLiteral.resolve(6)
assert PText.FromFunction(lambda: LIMIT).data is LIMIT.data and PText.FromFunction(lambda: LIMIT_DATA).data is LIMIT_DATA


def at_least(n):
    return lambda t: t.a >= n  # a closure: n is read when FromFunction runs


same_as(at_least(3), t.a.ge(3))


def beside():
    def helper(v):  # a nested def of several statements, beside the lambda
        w = v
        return w
    return helper(lambda t: t.a > 1)


same_as(beside(), t.a.gt(1))
assert V.OfAny(PText.FromFunction(lambda: 1 == 1.0)) is None  # the expression's rules, not Python's

# Collections, in the forms the Basic-to-Python translator writes, so its text reads back.
x = E.variable("x")  # each t.xs below is its own term, as in the lambdas
same_as(lambda t: all(x > 0 for x in t.xs), t.xs.all("x", x.gt(0)))
same_as(lambda t: any(x > 0 for x in t.xs if x < 5), t.xs.any("x", x.lt(5).and_(x.gt(0))))
same_as(lambda t: all(x > 0 for x in t.xs if x < 5 if x > 1), t.xs.all("x", x.lt(5).and_(x.gt(1)).implies(x.gt(0))))
same_as(lambda t: sum(1 for x in t.xs if x > 8.0) <= 2, t.xs.count_where("x", x.gt(8.0)).le(2))
same_as(lambda t: sum(1 for x in t.xs), t.xs.count_where("x", True))
same_as(lambda t: len(t.xs) > 0 and t.xs[0] == 1 and 3 in t.xs and 4 not in t.xs,
        t.xs.count().gt(0).and_(t.xs.item(0).eq(1)).and_(E.literal(3).in_(t.xs)).and_(E.literal(4).in_(t.xs).not_()))
same_as(lambda t: sum(t.xs) + max(t.xs) - min(t.xs), t.xs.sum().add(t.xs.max()).sub(t.xs.min()))
same_as(lambda t: all(any(y == x for y in t.ys) for x in t.xs), t.xs.all("x", t.ys.any("y", E.variable("y").eq(x))))
c = E.variable("c")
constraint = t.xs.count_where("c", c.gt(8.0)).le(2).and_(t.xs.max().le(12.0)).and_(t.xs.all("c", c.ge(2.0))).and_(
    t.xs.item(1).eq(1.0)).and_(E.literal(1.0).in_(t.xs)).and_(t.xs.count().gt(0))
written = ("sum(1 for c in t.xs if c > 8.0) <= 2 and max(t.xs) <= 12.0 and all(c >= 2.0 for c in t.xs) and "
           "t.xs[1] == 1.0 and 1.0 in t.xs and len(t.xs) > 0")
assert PText.ToText(T.between(E.DIALECT, P.DIALECT).forward(constraint)) == written
back = PText.FromFunction(lambda t: sum(1 for c in t.xs if c > 8.0) <= 2 and max(t.xs) <= 12.0 and
                          all(c >= 2.0 for c in t.xs) and t.xs[1] == 1.0 and 1.0 in t.xs and len(t.xs) > 0)
assert F.same(back.data, constraint.data)  # the translator's text reads back (each quantifier with its own c)


def shadowed(min):
    return lambda t: min(t.xs)  # min is not Python's own here


for function, message in [
    (lambda t: t.a / 2, "cannot convert 't.a / 2': not supported in an expression"),
    (lambda t: t.a if t else 1, "cannot convert 't.a if t else 1': not supported in an expression"),
    (lambda t: ~t.a, "cannot convert '~t.a': not supported in an expression"),
    (lambda t: t.a is not t, "cannot convert 't.a is not t': 'is' only tests whether a property is None"),
    (lambda: None, "cannot convert 'None': only native constants are literals"),
    (lambda: undefined_name, "cannot convert 'undefined_name': the name is not defined"),  # noqa: F821
    (lambda: linecache, "cannot convert 'linecache': a module is not a native value or an expression"),
    (lambda t=1: t, "only plain positional parameters can become variables"),
    (lambda: (lambda a, b: a)(1, 2), "cannot convert '(lambda a, b: a)(1, 2)': a let binds one name"),
    (lambda t: hasattr(t, 1), "cannot convert 'hasattr(t, 1)': hasattr takes an object and a property name"),
    (lambda t: hasattr(t, "x", y=1), """cannot convert "hasattr(t, 'x', y=1)": not supported in an expression"""),
    (lambda t: len(t, 1), "cannot convert 'len(t, 1)': len takes one collection"),
    (lambda t: all(t.xs), "cannot convert 'all(t.xs)': all takes a generator expression"),
    (lambda t: sum((x for x in t.xs)), "cannot convert 'sum((x for x in t.xs))': a quantifier is all"),
    (lambda t: all((x for x, y in t.xs)), "cannot convert 'all((x for x, y in t.xs))': a generator has one for"),
    (lambda t: any((x for x in t.xs for y in x)), "a generator has one for, over a name"),
    (lambda t: t.xs[0:1], "cannot convert 't.xs[0:1]': slices are not supported"),
    (lambda t: 1 < t.a is None, "cannot convert '1 < t.a is None': not supported in an expression"),
    (lambda len: len(1), "cannot convert 'len(1)': not supported in an expression"),
    (shadowed(max), "cannot convert 'min(t.xs)': not supported in an expression"),
    (lambda hasattr: hasattr(1, "x"), """cannot convert "hasattr(1, 'x')": not supported in an expression"""),
]:
    with raises(ValueError, match=message):
        PText.FromFunction(function)


def two_statements(t):
    t = t
    return t


def bare(t):
    return


for function in [two_statements, bare]:
    with raises(ValueError, match=f"the body of {function.__name__!r} must be a single return statement"):
        PText.FromFunction(function)
with raises(TypeError, match="expected a Python function, got builtin_function_or_method"):
    PText.FromFunction(len)
with raises(ValueError, match="the function's source is not available"):
    PText.FromFunction(eval("lambda: 1"))
linecache.cache["<unparsable>"] = (9, None, ["def (\n"], "<unparsable>")
with raises(ValueError, match="the function's source is not available"):
    PText.FromFunction(eval(compile("lambda: 1", "<unparsable>", "eval")))